# Tiered Gold-Standard HRD Labeling Demo

This notebook demonstrates the `TieredHRDLabeler` — a multi-modal concordance
labeling system that produces high-confidence training labels for HRD status.

**Tiers:**
- **Tier 1 (HRD-positive):** Biallelic HRR loss + high genomic scar (+ SBS3 if available)
- **Tier 2 (HRD-negative / HRP):** No HRR events + low genomic scar
- **Tier 3 (Ambiguous):** Excluded from training — monoallelic hits, intermediate scar, CDK12, etc.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Add repo root to path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'v2'))

from label_engineering.tiered_labels import TieredHRDLabeler

## 1. Load TCGA-BRCA Data

We load the same data files used in `prelim_analysis/softHRD_pipeline.ipynb`:
- `toga.breast.brca.status.txt` — Knijnenburg et al. BRCA annotation
- `tcga.hrdscore.xlsx` — HRD scores (LOH, TAI, LST, HRD-sum)

In [ ]:
DATA_DIR = os.path.join(REPO_ROOT, 'data')

brca_path = os.path.join(DATA_DIR, 'toga.breast.brca.status.txt')
hrd_path  = os.path.join(DATA_DIR, 'tcga.hrdscore.xlsx')

USE_REAL_DATA = os.path.exists(brca_path) and os.path.exists(hrd_path)

if USE_REAL_DATA:
    brca_df = pd.read_csv(brca_path, sep='\t', index_col=0)
    brca_df.index = brca_df.index.str.replace('.', '-', regex=False)
    hrd_df = pd.read_excel(hrd_path)
    print(f'Loaded BRCA status: {brca_df.shape}')
    print(f'Loaded HRD scores:  {hrd_df.shape}')
else:
    print('Data files not found — generating synthetic demo data.')
    print(f'  Looked for: {brca_path}')
    print(f'              {hrd_path}')

### Synthetic Data Fallback

If the real TCGA files are not present (data acquisition not yet complete),
we generate a synthetic cohort that mirrors the real data distribution to
demonstrate the labeler.

In [ ]:
def generate_synthetic_cohort(n=900, seed=42):
    """Generate a synthetic TCGA-BRCA-like cohort for demo purposes."""
    rng = np.random.default_rng(seed)
    samples = [f'TCGA-SYNTH-{i:04d}' for i in range(n)]

    # ~25% HRD in TCGA-BRCA
    hrd_frac = 0.24
    n_hrd = int(n * hrd_frac)

    # HRD-sum: bimodal — high for HRD, low for HRP, some intermediate
    hrd_sum = np.zeros(n)
    hrd_sum[:n_hrd] = rng.normal(55, 12, n_hrd).clip(10, 100).astype(int)
    hrd_sum[n_hrd:] = rng.exponential(10, n - n_hrd).clip(0, 70).astype(int)
    rng.shuffle(hrd_sum)

    # Event annotations
    event_brca1 = ['0'] * n
    event_brca2 = ['0'] * n
    event_rad51c = ['0'] * n
    event_palb2 = ['0'] * n

    # Detailed columns
    brca1_germ_bi = [0] * n
    brca1_germ_mono = [0] * n
    brca1_somatic_null = [0] * n
    brca1_deletion = [0] * n
    brca1_epi = [0] * n
    brca1_mRNA = [0] * n
    brca2_germ_bi = [0] * n
    brca2_germ_mono = [0] * n
    brca2_germ_undet = [0] * n
    brca2_somatic_null = [0] * n
    brca2_deletion = [0] * n
    rad51c_germ = [0] * n
    rad51c_deletion = [0] * n
    rad51c_epi = [0] * n
    rad51c_mRNA = [0] * n
    palb2_somatic = [0] * n
    palb2_germ = [0] * n
    tp53_somatic = [0] * n

    # Assign biallelic BRCA1/2 events to high-scar samples
    high_scar_idx = np.where(hrd_sum >= 42)[0]
    rng.shuffle(high_scar_idx)
    n_biallelic = int(len(high_scar_idx) * 0.7)
    for i in high_scar_idx[:n_biallelic]:
        gene_choice = rng.choice(['BRCA1', 'BRCA2', 'RAD51C', 'PALB2'], p=[0.45, 0.40, 0.08, 0.07])
        if gene_choice == 'BRCA1':
            event_brca1[i] = 'Bi-allelic-inactivation'
            brca1_germ_bi[i] = 1
        elif gene_choice == 'BRCA2':
            event_brca2[i] = 'Bi-allelic-inactivation'
            brca2_germ_bi[i] = 1
        elif gene_choice == 'RAD51C':
            event_rad51c[i] = 'Bi-allelic-inactivation'
            rad51c_germ[i] = 1
            rad51c_deletion[i] = 1
        else:
            event_palb2[i] = 'Bi-allelic-inactivation'
            palb2_germ[i] = 1
            palb2_somatic[i] = 1

    # Add BRCA1 methylation cases
    unmutated_high = [i for i in high_scar_idx[n_biallelic:] if event_brca1[i] == '0']
    for i in unmutated_high[:int(len(unmutated_high)*0.5)]:
        brca1_epi[i] = 1
        brca1_deletion[i] = 1  # with LOH
    for i in unmutated_high[int(len(unmutated_high)*0.5):int(len(unmutated_high)*0.7)]:
        brca1_epi[i] = 1
        brca1_deletion[i] = 0  # without LOH — will be ambiguous

    # Add monoallelic hits in intermediate-scar region
    inter_idx = np.where((hrd_sum >= 20) & (hrd_sum < 42))[0]
    for i in inter_idx[:int(len(inter_idx)*0.3)]:
        if rng.random() < 0.5:
            event_brca1[i] = '1'
            brca1_germ_mono[i] = 1
        else:
            event_brca2[i] = 'Bi-allelic-undetermined'
            brca2_germ_undet[i] = 1

    # TP53 somatic (common in TNBC, not HRD-specific)
    for i in range(n):
        if rng.random() < 0.35:
            tp53_somatic[i] = 1

    brca_df = pd.DataFrame({
        'BRCA1_somatic_null': brca1_somatic_null,
        'BRCA1_germ_bi_allelic': brca1_germ_bi,
        'BRCA1_germ_mono_allelic': brca1_germ_mono,
        'BRCA1_deletion': brca1_deletion,
        'BRCA1_epigenetic_silencing': brca1_epi,
        'BRCA1_mRNA': brca1_mRNA,
        'BRCA2_somatic_null': brca2_somatic_null,
        'BRCA2_germ_bi_allelic': brca2_germ_bi,
        'BRCA2_germ_undetermined': brca2_germ_undet,
        'BRCA2_germ_mono_allelic': brca2_germ_mono,
        'BRCA2_deletion': brca2_deletion,
        'RAD51C_germ': rad51c_germ,
        'RAD51C_deletion': rad51c_deletion,
        'RAD51C_epigenetic_silencing': rad51c_epi,
        'RAD51C_mRNA': rad51c_mRNA,
        'PALB2_somatic_null': palb2_somatic,
        'PALB2_germ': palb2_germ,
        'TP53_somatic': tp53_somatic,
        'event.BRCA1': event_brca1,
        'event.BRCA2': event_brca2,
        'event.RAD51C': event_rad51c,
        'event.PALB2': event_palb2,
        'event.All Events': ['0'] * n,
    }, index=samples)

    loh = (hrd_sum * rng.uniform(0.25, 0.40, n)).astype(int)
    tai = (hrd_sum * rng.uniform(0.25, 0.40, n)).astype(int)
    lst = hrd_sum - loh - tai
    lst = lst.clip(0)

    hrd_scores_df = pd.DataFrame({
        'sample': samples,
        'HRD': loh,
        'Telomeric AI': tai,
        'LST': lst,
        'HRD-sum': hrd_sum.astype(int),
    })

    return brca_df, hrd_scores_df

if not USE_REAL_DATA:
    brca_df, hrd_df = generate_synthetic_cohort()
    print(f'Synthetic BRCA status: {brca_df.shape}')
    print(f'Synthetic HRD scores:  {hrd_df.shape}')

## 2. Apply Tiered Labeling

In [ ]:
labeler = TieredHRDLabeler(
    gis_positive_threshold=42,
    gis_negative_threshold=20,
    gloh_threshold=0.16,
    sbs3_threshold=0.06,
)

tiered = labeler.label_tcga_cohort(brca_df, hrd_df)
print(f'Labeled {len(tiered)} samples')
tiered.head(10)

## 3. Tier Distribution

In [ ]:
tier_counts = tiered['tier'].value_counts().sort_index()
print('Samples per tier:')
for t, c in tier_counts.items():
    lbl = {1: 'HRD-positive', 2: 'HRP (negative)', 3: 'Ambiguous'}[t]
    pct = 100 * c / len(tiered)
    print(f'  Tier {t} ({lbl}): {c}  ({pct:.1f}%)')

print(f'\nUsable for training (Tier 1 + 2): {tier_counts.get(1,0) + tier_counts.get(2,0)}')
print(f'Excluded (Tier 3): {tier_counts.get(3,0)}')

In [ ]:
colors = {1: '#d62728', 2: '#2ca02c', 3: '#7f7f7f'}
labels_map = {1: 'Tier 1: HRD+', 2: 'Tier 2: HRP', 3: 'Tier 3: Ambiguous'}

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    [labels_map[t] for t in tier_counts.index],
    tier_counts.values,
    color=[colors[t] for t in tier_counts.index],
)
for bar, val in zip(bars, tier_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontsize=11)
ax.set_ylabel('Number of samples')
ax.set_title('Tiered Gold-Standard Label Distribution')
sns.despine()
plt.tight_layout()
plt.show()

## 4. Compare with Simple HRD-sum >= 42 Threshold

The existing approach labels samples as HRD if HRD-sum >= 42 and HR if < 42.
Let's see how many samples get reclassified by the tiered system.

In [ ]:
# Create the simple binary labels
existing = tiered['HRD-sum'].apply(lambda x: 'HRD' if x is not None and x >= 42 else 'HRP')
existing.name = 'HRD_status_base'
print('Simple threshold counts:')
print(existing.value_counts())

In [ ]:
comparison = labeler.compare_with_existing_labels(tiered, existing, 'HRD_status_base')

print(f"Total samples: {comparison['total_samples']}")
print(f"Tier 1+2 (definite labels): {comparison['tier12_samples']}")
print(f"Concordant: {comparison['concordant']}")
print(f"Discordant: {comparison['discordant']}")
print(f"Concordance rate (Tier 1+2): {comparison['concordance_rate']:.1%}")
print(f"Moved to ambiguous (Tier 3): {len(comparison['moved_to_ambiguous'])}")

In [ ]:
reclass = comparison['reclassified']
if len(reclass) > 0:
    print(f'\n--- Reclassified samples (n={len(reclass)}) ---')
    print(reclass[['tier', 'label', 'existing_label', 'HRD-sum', 'reasons']].to_string())
else:
    print('No reclassified samples between Tier 1/2 labels.')

In [ ]:
ambig = comparison['moved_to_ambiguous']
print(f'\n--- Samples moved to Tier 3 (ambiguous) that had existing labels ---')
print(f'Were HRD: {(ambig["existing_label"] == "HRD").sum()}')
print(f'Were HRP: {(ambig["existing_label"] == "HRP").sum()}')
print(f'\nTop reasons for exclusion:')
# Count reasons
from collections import Counter
all_reasons = []
for r in ambig['reasons']:
    all_reasons.extend([x.strip() for x in r.split(';')])
for reason, count in Counter(all_reasons).most_common(10):
    print(f'  {count:4d}  {reason}')

## 5. Visualize HRD-sum Distribution by Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram colored by tier
ax = axes[0]
for tier_val in [2, 3, 1]:
    subset = tiered[tiered['tier'] == tier_val]
    ax.hist(subset['HRD-sum'].dropna(), bins=40, alpha=0.6,
            color=colors[tier_val], label=labels_map[tier_val], range=(0, 100))
ax.axvline(42, color='black', linestyle='--', linewidth=1.5, label='GIS=42 threshold')
ax.axvline(20, color='gray', linestyle=':', linewidth=1.5, label='GIS=20 threshold')
ax.set_xlabel('HRD-sum')
ax.set_ylabel('Number of samples')
ax.set_title('HRD-sum distribution by tier')
ax.legend(fontsize=9)

# Right: strip/swarm by tier
ax = axes[1]
plot_df = tiered[['tier', 'HRD-sum']].dropna().copy()
plot_df['Tier'] = plot_df['tier'].map(labels_map)
sns.stripplot(data=plot_df, x='Tier', y='HRD-sum', hue='Tier',
              palette={labels_map[k]: v for k, v in colors.items()},
              alpha=0.4, jitter=0.3, ax=ax, legend=False)
ax.axhline(42, color='black', linestyle='--', linewidth=1)
ax.axhline(20, color='gray', linestyle=':', linewidth=1)
ax.set_ylabel('HRD-sum')
ax.set_title('HRD-sum by tier (each dot = 1 sample)')

sns.despine()
plt.tight_layout()
plt.show()

## 6. Tier Breakdown by Gene Event

In [ ]:
event_cols = ['event.BRCA1', 'event.BRCA2', 'event.RAD51C', 'event.PALB2']
available_cols = [c for c in event_cols if c in tiered.columns]

if available_cols:
    for col in available_cols:
        print(f'\n--- {col} ---')
        ct = pd.crosstab(tiered[col], tiered['tier'], margins=True)
        ct.columns = [f'Tier {c}' if c != 'All' else 'Total' for c in ct.columns]
        print(ct)
else:
    print('Event columns not found in tiered output.')

## Summary

The tiered labeling system:

1. **Selects high-confidence positives** — only samples with both biallelic HRR loss AND high genomic scar get Tier 1
2. **Selects high-confidence negatives** — only samples with no HRR events AND low scar get Tier 2
3. **Excludes the messy middle** — monoallelic hits, intermediate scar, methylation without LOH confirmation, etc.

This gives us a cleaner training set where labels are supported by multiple orthogonal lines of evidence,
at the cost of reduced sample size (Tier 3 samples are excluded from training).